# AstroCLIP Teaching Exercises

Complete the TODOs below to reproduce the full pipeline.

0## 0. Environment Setup
Import the libraries, set `ASTROCLIP_ROOT`, and detect the device.

In [2]:
# Imports and environment setup
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from torch.utils.data import Dataset, DataLoader, TensorDataset
import umap
from datasets import load_from_disk

from astroclip.data.datamodule import AstroClipCollator
from teaching_scripts.data_utils import (
    build_image_dataloader,
    build_spectrum_dataloader,
    build_multimodal_dataloader,
)
from teaching_scripts.models import ImageAutoencoder, SpectrumAutoencoder, SmallCLIPModel

ASTROCLIP_ROOT = Path(os.environ.get("ASTROCLIP_ROOT", "/pbs/throng/training/astroinfo2025/data/AstroCLIP_data")).resolve()
ASTROCLIP_ROOT.mkdir(parents=True, exist_ok=True)

dataset_path = ASTROCLIP_ROOT / "astroclip_subset_tiny"
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset not found at {dataset_path}. Prepare the subset first.")

ds = load_from_disk(dataset_path)
print({split: len(ds[split]) for split in ds})

collator = AstroClipCollator(center_crop=144)

image_train_loader = build_image_dataloader(ds["train"], batch_size=128, shuffle=True, num_workers=0)
image_val_loader   = build_image_dataloader(ds["test"],  batch_size=128, shuffle=False, num_workers=0)

spectrum_train_loader = build_spectrum_dataloader(ds["train"], batch_size=256, shuffle=True, num_workers=0)
spectrum_val_loader   = build_spectrum_dataloader(ds["test"],  batch_size=256, shuffle=False, num_workers=0)

multimodal_train_loader = build_multimodal_dataloader(ds["train"], batch_size=256, shuffle=True, num_workers=0)
multimodal_val_loader   = build_multimodal_dataloader(ds["test"],  batch_size=256, shuffle=False, num_workers=0)

device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)
print("Using device:", device)

plt.rcParams["figure.facecolor"] = "white"

def latest_version(log_dir: Path, run_name: str) -> Path:
    run_dir = log_dir / run_name
    candidates = sorted(run_dir.glob("version_*"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        run_dir.mkdir(parents=True, exist_ok=True)
        (run_dir / "version_0").mkdir(exist_ok=True)
        candidates = [run_dir / "version_0"]
    return candidates[-1]


/Users/marchuertascompany/soft/miniforge3/envs/astroclip-mac/lib/python3.10/site-packages/lightning/fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/Users/marchuertascompany/soft/miniforge3/envs/astroclip-mac/lib/python3.10/site-packages/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/marchuertascompany/soft/miniforge3/envs/astroclip-mac/lib/python3.10/site-packages/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/marchuertascompany/soft/miniforge3/envs/astroclip-mac/lib/python3.10/site-packages/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is no

OSError: [Errno 30] Read-only file system: '/pbs'

## 1. Image & Spectrum Encoders
Create the image autoencoder and the new spectrum transformer.

In [1]:
# Student-defined models (fill in the TODOs)

class StudentImageAutoencoder(nn.Module):
    def __init__(self, embed_dim: int = 256):
        super().__init__()
        # Simple conv encoder → linear bottleneck
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(128 * 18 * 18, embed_dim),
        )
        # Linear + deconv decoder
        self.decoder_fc = nn.Sequential(
            nn.Linear(embed_dim, 128 * 18 * 18),
            nn.ReLU(inplace=True),
        )
        self.decoder_conv = nn.Sequential(
            nn.Unflatten(1, (128, 18, 18)),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder_conv(self.decoder_fc(z))

    def forward(self, x):
        return self.decode(self.encode(x))


class StudentSpectrumAutoencoder(nn.Module):
    def __init__(self, input_dim: int = 7781, embed_dim: int = 256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, embed_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, input_dim),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        return self.decode(self.encode(x))


class StudentSpectrumTransformer(nn.Module):
    def __init__(
        self,
        input_dim: int = 7781,
        patch_size: int = 16,
        embed_dim: int = 256,
        num_layers: int = 2,
        num_heads: int = 4,
    ):
        super().__init__()
        if input_dim % patch_size != 0:
            raise ValueError("input_dim must be divisible by patch_size")
        self.patch_size = patch_size
        self.num_patches = input_dim // patch_size

        # Patch embedding + positional encoding
        self.patch_embed = nn.Linear(patch_size, embed_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, self.num_patches, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            batch_first=True,
            dropout=0.1,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self.reconstruction = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Linear(embed_dim * 2, input_dim),
        )

    def encode(self, spectrum: torch.Tensor):
        patches = spectrum.view(spectrum.size(0), self.num_patches, self.patch_size)
        tokens = self.patch_embed(patches) + self.positional_encoding
        encoded = self.transformer(tokens)
        pooled = encoded.mean(dim=1)
        return self.norm(pooled)

    def forward(self, spectrum: torch.Tensor):
        return self.reconstruction(self.encode(spectrum))


class StudentCLIP(nn.Module):
    def __init__(self, image_encoder, spectrum_encoder, projection_dim: int = 256, temperature: float = 0.07):
        super().__init__()
        self.image_encoder = image_encoder
        self.spectrum_encoder = spectrum_encoder
        self.temperature = temperature

        with torch.no_grad():
            img_sample = torch.randn(1, 3, 144, 144)
            img_dim = image_encoder.encode(img_sample).shape[-1]
            spec_sample = torch.randn(1, 7781)
            spec_dim = spectrum_encoder.encode(spec_sample).shape[-1]

        self.img_proj = nn.Linear(img_dim, projection_dim)
        self.spec_proj = nn.Linear(spec_dim, projection_dim)

    def encode_image(self, images):
        features = self.image_encoder.encode(images)
        proj = self.img_proj(features)
        return F.normalize(proj, dim=-1)

    def encode_spectrum(self, spectra):
        if hasattr(self.spectrum_encoder, "encode"):
            features = self.spectrum_encoder.encode(spectra)
        else:
            features = self.spectrum_encoder(spectra)
        proj = self.spec_proj(features)
        return F.normalize(proj, dim=-1)

    def forward(self, images, spectra):
        return self.encode_image(images), self.encode_spectrum(spectra)


NameError: name 'nn' is not defined

### Training or Loading
Train from scratch or load checkpoints for both encoders.

In [ ]:
# Train or load helpers for the encoders

def train_with_lightning(model, ckpt_path, train_loader, val_loader, *, max_epochs, logger_name):
    logger = CSVLogger(save_dir=str(ASTROCLIP_ROOT / "logs"), name=logger_name)
    checkpoint = ModelCheckpoint(
        dirpath=ckpt_path.parent,
        filename=ckpt_path.stem,
        save_last=True,
        save_top_k=1,
        monitor="val_loss",
        mode="min",
    )
    trainer = L.Trainer(
        accelerator="auto",
        devices="auto",
        max_epochs=max_epochs,
        log_every_n_steps=25,
        callbacks=[checkpoint],
        logger=logger,
    )
    trainer.fit(model, train_loader, val_loader)
    trainer.save_checkpoint(ckpt_path)
    return model

image_ckpt_path = ASTROCLIP_ROOT / "models/student_image_autoencoder.ckpt"
spectrum_ckpt_path = ASTROCLIP_ROOT / "models/student_spectrum_autoencoder.ckpt"

if image_ckpt_path.exists():
    image_model = StudentImageAutoencoder.load_from_checkpoint(image_ckpt_path)
else:
    image_model = StudentImageAutoencoder()
    image_model = train_with_lightning(image_model, image_ckpt_path, image_train_loader, image_val_loader, max_epochs=8, logger_name="student_image_autoencoder")

if spectrum_ckpt_path.exists():
    spectrum_model = StudentSpectrumAutoencoder.load_from_checkpoint(spectrum_ckpt_path)
else:
    spectrum_model = StudentSpectrumAutoencoder()
    spectrum_model = train_with_lightning(spectrum_model, spectrum_ckpt_path, spectrum_train_loader, spectrum_val_loader, max_epochs=12, logger_name="student_spectrum_autoencoder")


## 2. CLIP Alignment & Embedding Exploration
Train/Load CLIP model and run UMAP.

In [ ]:
class StudentCLIP(torch.nn.Module):
    def __init__(self, image_encoder, spectrum_encoder, projection_dim=256, temperature=0.07):
        super().__init__()
        self.image_encoder = image_encoder
        self.spectrum_encoder = spectrum_encoder
        self.img_proj = nn.Linear(image_encoder.embed_dim, projection_dim)
        self.spec_proj = nn.Linear(spectrum_encoder.embed_dim, projection_dim)
        self.temperature = temperature

    def encode_image(self, images):
        features = self.image_encoder.encode(images)
        proj = self.img_proj(features)
        return F.normalize(proj, dim=-1)

    def encode_spectrum(self, spectra):
        if hasattr(self.spectrum_encoder, 'encode'):
            features = self.spectrum_encoder.encode(spectra)
        else:
            features = self.spectrum_encoder(spectra)
        proj = self.spec_proj(features)
        return F.normalize(proj, dim=-1)

    def forward(self, images, spectra):
        return self.encode_image(images), self.encode_spectrum(spectra)

    def training_step(self, batch, batch_idx):
        images, spectra = batch['image'], batch['spectrum']
        img, spec = self(images, spectra)
        logits = (img @ spec.T) / self.temperature
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, spectra = batch['image'], batch['spectrum']
        img, spec = self(images, spectra)
        logits = (img @ spec.T) / self.temperature
        targets = torch.arange(logits.size(0), device=logits.device)
        loss = (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2
        self.log('val_loss', loss, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=5e-4, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}


In [ ]:
# Generate embeddings for the validation split and visualise with UMAP
image_embeddings, spectrum_embeddings, redshift = [], [], []

with torch.no_grad():
    for batch in multimodal_val_loader:
        imgs = batch["image"].to(device)
        specs = batch["spectrum"].to(device)
        img_embeds, spec_embeds = clip_model(imgs, specs)
        image_embeddings.append(img_embeds.cpu().numpy())
        spectrum_embeddings.append(spec_embeds.cpu().numpy())
        redshift.append(batch["redshift"].numpy())

image_embeddings = np.concatenate(image_embeddings)
spectrum_embeddings = np.concatenate(spectrum_embeddings)
redshift = np.concatenate(redshift)

reducer = umap.UMAP(random_state=42)
joint = reducer.fit_transform(np.concatenate([image_embeddings, spectrum_embeddings], axis=0))
labels = np.concatenate([np.zeros(len(image_embeddings)), np.ones(len(spectrum_embeddings))])

plt.figure(figsize=(6, 5))
plt.scatter(joint[labels == 0, 0], joint[labels == 0, 1], s=5, alpha=0.5, label="Images")
plt.scatter(joint[labels == 1, 0], joint[labels == 1, 1], s=5, alpha=0.5, label="Spectra")
plt.legend()
plt.title("UMAP of joint embedding space")
plt.show()


## 3. Regression Head
Use the embeddings to train a regression model.

In [ ]:
# Regression head on image embeddings
train_size = int(0.8 * len(image_embeddings))
train_emb = image_embeddings[:train_size]
val_emb = image_embeddings[train_size:]
train_z = redshift[:train_size]
val_z = redshift[train_size:]

train_ds = TensorDataset(torch.from_numpy(train_emb).float(), torch.from_numpy(train_z).float())
val_ds = TensorDataset(torch.from_numpy(val_emb).float(), torch.from_numpy(val_z).float())
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=256, shuffle=False)

class StudentRegressor(nn.Module):
    def __init__(self, input_dim, hidden=256):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        return self.model(x).squeeze(-1)

regressor = StudentRegressor(train_emb.shape[1]).to(device)
optimizer = torch.optim.AdamW(regressor.parameters(), lr=3e-4, weight_decay=1e-5)
criterion = nn.MSELoss()

train_losses, val_losses = [], []
for epoch in range(40):
    regressor.train()
    running = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        preds = regressor(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * xb.size(0)
    train_loss = running / len(train_dl.dataset)

    regressor.eval()
    running = 0.0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            preds = regressor(xb)
            loss = criterion(preds, yb)
            running += loss.item() * xb.size(0)
    val_loss = running / len(val_dl.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d}: train={train_loss:.4f}, val={val_loss:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Image-only embeddings redshift loss")
plt.legend()
plt.show()

regressor.eval()
preds_img = []
with torch.no_grad():
    for xb, _ in val_dl:
        preds_img.extend(regressor(xb.to(device)).cpu().numpy())
preds_img = np.array(preds_img)
true_img = val_z[: len(preds_img)]

plt.figure(figsize=(5, 5))
plt.scatter(true_img, preds_img, s=8, alpha=0.6)
lims = [true_img.min(), true_img.max()]
plt.plot(lims, lims, "--", color="gray")
plt.xlabel("True redshift")
plt.ylabel("Predicted redshift")
plt.title("Image-only toy embeddings: predicted vs true")
plt.show()


## 4. CNN Baseline
Train a CNN directly on images for comparison.

In [ ]:
# Image-only CNN baseline
class ImageRedshiftDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        record = dict(self.dataset[idx])
        image = torch.tensor(np.asarray(record["image"]), dtype=torch.float32).permute(2, 0, 1)
        redshift = torch.tensor(float(record["redshift"]), dtype=torch.float32)
        return image, redshift

cnn_train_dl = DataLoader(ImageRedshiftDataset(ds["train"]), batch_size=128, shuffle=True, num_workers=0)
cnn_val_dl   = DataLoader(ImageRedshiftDataset(ds["test"]),  batch_size=128, shuffle=False, num_workers=0)

class StudentImageCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.regressor(features).squeeze(-1)

cnn_model = StudentImageCNN().to(device)
optimizer_cnn = torch.optim.AdamW(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion_cnn = nn.MSELoss()

cnn_train_losses, cnn_val_losses = [], []
for epoch in range(5):
    cnn_model.train()
    running = 0.0
    for xb, yb in tqdm(cnn_train_dl, desc=f"Epoch {epoch+1}/5", leave=False):
        xb, yb = xb.to(device), yb.to(device)
        optimizer_cnn.zero_grad(set_to_none=True)
        preds = cnn_model(xb)
        loss = criterion_cnn(preds, yb)
        loss.backward()
        optimizer_cnn.step()
        running += loss.item() * xb.size(0)
    train_loss = running / len(cnn_train_dl.dataset)

    cnn_model.eval()
    running = 0.0
    with torch.no_grad():
        for xb, yb in cnn_val_dl:
            xb, yb = xb.to(device), yb.to(device)
            preds = cnn_model(xb)
            loss = criterion_cnn(preds, yb)
            running += loss.item() * xb.size(0)
    val_loss = running / len(cnn_val_dl.dataset)
    cnn_train_losses.append(train_loss)
    cnn_val_losses.append(val_loss)
    tqdm.write(f"Epoch {epoch+1:02d}: train={train_loss:.4f}, val={val_loss:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(cnn_train_losses, label="train")
plt.plot(cnn_val_losses, label="val")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Image-only CNN redshift loss")
plt.legend()
plt.show()

cnn_model.eval()
cnn_preds, cnn_truth = [], []
with torch.no_grad():
    for xb, yb in cnn_val_dl:
        preds = cnn_model(xb.to(device)).cpu().numpy()
        cnn_preds.append(preds)
        cnn_truth.append(yb.numpy())
cnn_preds = np.concatenate(cnn_preds)
cnn_truth = np.concatenate(cnn_truth)

plt.figure(figsize=(5, 5))
plt.scatter(cnn_truth, cnn_preds, s=8, alpha=0.5)
lims = [cnn_truth.min(), cnn_truth.max()]
plt.plot(lims, lims, '--', color='gray')
plt.xlabel('True redshift')
plt.ylabel('Predicted redshift')
plt.title('Image-only CNN predictions')
plt.show()


## 5. Compare Results
Plot predicted vs. true redshift for CLIP and baseline; discuss.

In [ ]:
# Compare results
from sklearn.metrics import mean_squared_error

# CLIP regressor predictions already stored in preds_img / true_img
clip_mse = mean_squared_error(true_img, preds_img)

cnn_mse = mean_squared_error(cnn_truth, cnn_preds)

print(f"CLIP embedding regressor MSE: {clip_mse:.4f}")
print(f"CNN baseline MSE: {cnn_mse:.4f}")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(true_img, preds_img, s=8, alpha=0.5)
lims = [true_img.min(), true_img.max()]
plt.plot(lims, lims, '--', color='gray')
plt.xlabel('True redshift')
plt.ylabel('Predicted redshift')
plt.title('CLIP embeddings')

plt.subplot(1, 2, 2)
plt.scatter(cnn_truth, cnn_preds, s=8, alpha=0.5)
lims = [cnn_truth.min(), cnn_truth.max()]
plt.plot(lims, lims, '--', color='gray')
plt.xlabel('True redshift')
plt.ylabel('Predicted redshift')
plt.title('Image-only CNN')

plt.tight_layout()
plt.show()
